# Warm-start Training on Colab (GPU)

Fine-tune from the pretrained `rabah2026/wav2vec2-large-xlsr-53-arabic-quran-v_final`
on Quran audio, using the project's vocab + the **bootstrap→finetune** staging trainer.

**Bahasa Indonesia:** Fine-tuning dari model rabah2026 dengan vocab + teks quran.com,
memakai trainer staging (bootstrap head-only → finetune encoder terakhir).

Run cells top-to-bottom on a **GPU (T4)** runtime: *Runtime → Change runtime type → GPU*.
Audio, dataset, and checkpoints persist on **Google Drive**, so a Colab disconnect
loses nothing — just re-mount Drive and re-run Download → Build → Train (cached/resumed).

In [ ]:
# GPU check — must show a T4 (or better) before continuing
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Runtime is NOT GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
# Env: keep the HF model/dataset cache on runtime disk (/content), not Drive
import os
os.environ.setdefault('HF_HOME', '/content/hf_cache')
# os.environ['HF_TOKEN'] = 'hf_xxx'   # uncomment ONLY to push the trained model to the Hub

In [ ]:
# Mount Google Drive — the warmstart config writes audio/dataset/checkpoints here
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the project onto Colab.
# Option A: clone from GitHub (set YOUR repo URL below).
# Option B: skip this cell and instead upload the project folder to /content/model-asr-quran.
import os
REPO = '/content/model-asr-quran'
GITHUB_URL = 'https://github.com/<your-user>/model-asr-quran.git'   # <-- set this

if not os.path.isdir(REPO):
    !git clone {GITHUB_URL} {REPO}
%cd {REPO}
!git log -1 --oneline || echo '(not a git checkout — uploaded folder)'

## 1. Environment setup

Install Colab deps (keeps PyTorch CUDA GPU-ready) + the project (editable, `--no-deps`
so pip won't swap PyTorch), then pre-flight checks (GPU / ffmpeg / disk).

In [ ]:
!bash scripts/install_colab_deps.sh
!python -m pip install -q -e . --no-deps
!python scripts/cloud_preflight.py

## 2. Data + training

`configs/cloud_warmstart.yaml` points at:
- **base**: `rabah2026/wav2vec2-large-xlsr-53-arabic-quran-v_final` (warm-start)
- **data**: everyayah audio (Husary + Minshawy) + quran.com text → Google Drive
- **trainer**: `scripts/train_manual.py` → `manual_train`, with `auto_stage` (bootstrap head-only → finetune last encoder layers)

Re-run any cell to resume: download/build/vocab are cached, and training resumes from
`latest` (the config sets `resume_from: latest`).

In [ ]:
# Download everyayah audio + quran.com text (~20-40 min, resumable) → Drive
!python scripts/download.py --config configs/cloud_warmstart.yaml

In [ ]:
# Build the HF dataset (join audio + quran.com text, by_surah split)
!python scripts/build.py --config configs/cloud_warmstart.yaml

In [ ]:
# Build the diacritics-aware CTC vocab from the quran.com corpus
!python scripts/build_vocab.py --config configs/cloud_warmstart.yaml

In [ ]:
# Warm-start training — LONG-RUNNING (hours on T4). Re-run to resume after a disconnect.
# IMPORTANT: use train_manual.py (staging + head re-init), NOT scripts/train.py.
!python scripts/train_manual.py --config configs/cloud_warmstart.yaml

## 3. After training

Final + best checkpoints are on Drive at
`/content/drive/MyDrive/quran_asr/checkpoints_warmstart/` (`final/`, `best/`).
Copy `final/` to your backend to serve it, or run the inference sanity check below.

In [ ]:
# Sanity: run the corrector on Al-Fatiha 1:1 with the trained final checkpoint
FINAL = '/content/drive/MyDrive/quran_asr/checkpoints_warmstart/final'
AUDIO = '/content/drive/MyDrive/quran_asr/data/raw/audio/Husary_128kbps_Mujawwad/001001.wav'
!python scripts/correct.py --model-dir {FINAL} --audio {AUDIO} \
    --text 'بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ'